# Lekcja 5: EDA i klasyfikacja na danych pingwinów (Palmer Penguins)

W tej lekcji zrobimy **pełny, ale prosty workflow data science**:
1. instalacja bibliotek,
2. wczytanie danych,
3. podstawowe **Exploratory Data Analysis (EDA)**,
4. wizualizacja danych,
5. jeden prosty model klasyfikacji (**kNN - k-Nearest Neighbors**).

**Główny cel:** będziemy przewidywać, **do jakiego gatunku należy pingwin** na podstawie kilku pomiarów (np. długości dzioba, długości płetwy, masy ciała).

Mamy dane opisujące wiele różnych pingwinów. Najpierw spróbujemy odkryć zależności w danych (EDA), a potem wykorzystamy je do prostego modelowania (classification).

> Wszystko robimy krok po kroku. W wielu komórkach są `TODO`, ale wymagają tylko małych zmian.

## Dla kogo jest ten notebook?

Dla klasy 7 (poziom podstawowy).

- Nie musisz rozumieć wszystkiego od razu.
- Najpierw **uruchamiasz**, potem **czytasz wynik**, a dopiero na końcu **modyfikujesz** drobiazgi.
- To jest tutorial: prowadzi za rękę.

## Jakie gatunki pingwinów mamy w danych?

W zbiorze Palmer Penguins występują 3 gatunki:
- **Adelie**
- **Chinstrap**
- **Gentoo**

Poniżej przykładowe zdjęcia (Wikimedia Commons):

| Gatunek | Przykładowe zdjęcie |
|---|---|
| Adelie | ![Pingwin Adelie](https://upload.wikimedia.org/wikipedia/commons/e/e7/Adelie_Penguin.jpg) |
| Chinstrap | ![Pingwin Chinstrap](https://upload.wikimedia.org/wikipedia/commons/6/6f/Chinstrap_Penguin.jpg) |
| Gentoo | ![Pingwin Gentoo](https://upload.wikimedia.org/wikipedia/commons/0/0f/Gentoo_Penguin_1.jpg) |

> Jeśli obrazy się nie wyświetlą (np. brak internetu), kontynuuj lekcję — nie wpływa to na działanie kodu.

## 0) Instalacja bibliotek (installation)

**Co to jest instalacja?**

Instalacja bibliotek to dołożenie gotowych narzędzi do Pythona.
Dzięki temu nie piszemy wszystkiego od zera.

W Jupyterze najlepiej instalować przez `%pip install ...` w komórce.

> Jeśli biblioteki są już zainstalowane, komenda po prostu to potwierdzi.

In [ ]:
# TODO: Uruchom tę komórkę tylko raz na początku zajęć.
%pip install -q pandas matplotlib seaborn scikit-learn palmerpenguins

## 1) Import bibliotek (imports)

Import to włączenie biblioteki do aktualnego notebooka.

In [ ]:
# TODO: Uruchom komórkę bez zmian.
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from palmerpenguins import load_penguins
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Ustawienia wykresów
sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (8, 5)

## 2) Wczytanie danych

Dane pingwinów mają m.in. takie cechy: długość dzioba, głębokość dzioba, długość płetwy, masa ciała.

Naszym celem będzie przewidzieć gatunek pingwina (`species`).

In [ ]:
# TODO: Uruchom komórkę.
df = load_penguins()
print('Kształt tabeli (shape):', df.shape)
df.head()

## 3) Szybki przegląd danych (quick EDA)

Tutaj sprawdzimy:
- nazwy kolumn,
- typy danych,
- brakujące wartości.

In [ ]:
# TODO: Uruchom i przeczytaj wynik.
print('Kolumny:')
print(df.columns.tolist())
print('\nTypy danych:')
print(df.dtypes)
print('\nBrakujące wartości w kolumnach:')
print(df.isna().sum())

## 4) Przygotowanie prostego zbioru do modelu

Na start użyjemy 4 cech liczbowych i celu `species`.

Braki danych usuniemy (`dropna`) - to najprostsza metoda na początek.

In [ ]:
# TODO: Uruchom komórkę.
features = ['bill_length_mm', 'bill_depth_mm', 'flipper_length_mm', 'body_mass_g']
target = 'species'

data = df[features + [target]].dropna().copy()
print('Nowy kształt danych po dropna:', data.shape)
data.head()

## 5) Statystyki opisowe (descriptive statistics)

Policzymy podstawowe miary:

- **Średnia (mean)** - taka typowa wartość, ale wrażliwa na skrajne przypadki.
- **Mediana (median)** - wartość środkowa po uporządkowaniu; bardziej odporna na skrajne wartości.
- **Odchylenie standardowe (standard deviation, std)** - mówi, jak bardzo wyniki są rozrzucone.
  - małe std -> wartości są blisko siebie,
  - duże std -> wartości są bardziej rozsiane.

In [ ]:
# TODO: Uruchom komórkę i porównaj mean vs median.
statystyki = data[features].agg(['mean', 'median', 'std']).T
statystyki

### Mini-interpretacja

Jeśli dla jakiejś cechy **średnia i mediana są podobne**, rozkład bywa dość symetryczny.
Jeśli są mocno różne, możliwe są wartości skrajne albo skośny rozkład.

In [ ]:
# TODO (prosty): Wybierz jedną cechę i wypisz jej mean, median, std.
wybrana_cecha = 'body_mass_g'  # możesz zmienić np. na 'bill_length_mm'

print('Wybrana cecha:', wybrana_cecha)
print('mean   =', data[wybrana_cecha].mean())
print('median =', data[wybrana_cecha].median())
print('std    =', data[wybrana_cecha].std())

## 6) Wizualizacje (visualizations)

Na razie 3 klasyczne wykresy:
1. histogram,
2. boxplot,
3. scatter plot z kolorem gatunku.

In [ ]:
# 6.1 Histogram
# TODO: Uruchom. Potem możesz zmienić kolumnę w x=.
sns.histplot(data=data, x='flipper_length_mm', kde=True)
plt.title('Histogram: flipper_length_mm')
plt.show()

In [ ]:
# 6.2 Boxplot
# TODO: Uruchom.
sns.boxplot(data=data, x='species', y='body_mass_g')
plt.title('Boxplot: masa ciała wg gatunku')
plt.show()

In [ ]:
# 6.3 Scatter plot
# TODO: Uruchom.
sns.scatterplot(
    data=data,
    x='bill_length_mm',
    y='bill_depth_mm',
    hue='species'
)
plt.title('Zależność: długość dzioba vs głębokość dzioba')
plt.show()

### TODO ucznia: mikro-zmiana w wizualizacji

Zmodyfikuj tylko dwie linie w następnej komórce:
- ustaw inną parę cech na osiach,
- zostaw `hue='species'`.

In [ ]:
# TODO: Zmień x i y na inne kolumny z listy 'features'.
# Podpowiedź: np. x='flipper_length_mm', y='body_mass_g'

sns.scatterplot(
    data=data,
    x='flipper_length_mm',   # TODO: zmień
    y='body_mass_g',         # TODO: zmień
    hue='species'
)
plt.title('Twoja wersja scatter plot')
plt.show()

## 7) Prosty model klasyfikacji: kNN

Użyjemy jednego modelu: **k-Nearest Neighbors (kNN)**.

Intuicja:
- model patrzy na najbliższe punkty,
- głosuje, do jakiego gatunku należy nowy pingwin.

`k` = liczba sąsiadów użytych do głosowania.

In [ ]:
# TODO: Uruchom komórkę bez zmian.
X = data[features]
y = data[target]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print('X_train:', X_train.shape)
print('X_test :', X_test.shape)

In [ ]:
# TODO: Tu jest najważniejsza mikro-zmiana ucznia.
# Zmień k z 5 na 3 lub 7 i sprawdź, czy accuracy się zmieni.

k = 5  # TODO: zmień na 3 albo 7
model = KNeighborsClassifier(n_neighbors=k)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
acc = accuracy_score(y_test, y_pred)

print(f'Accuracy dla k={k}: {acc:.3f}')

## 8) Ocena modelu (evaluation)

- **Accuracy**: jaki procent przykładów model zaklasyfikował poprawnie.
- **Confusion matrix**: gdzie model się myli (między którymi klasami).
- **Classification report**: dokładniejsze metryki dla każdej klasy.

In [ ]:
# TODO: Uruchom.
cm = confusion_matrix(y_test, y_pred, labels=sorted(y.unique()))
cm_df = pd.DataFrame(cm, index=sorted(y.unique()), columns=sorted(y.unique()))
cm_df

In [ ]:
# TODO: Uruchom.
print(classification_report(y_test, y_pred))

## 9) Zadanie końcowe (bardzo małe kroki)

1. Zmień `k` na inną wartość (np. 3 i 7).
2. Zapisz accuracy dla każdej wartości.
3. Napisz 1-2 zdania: która wartość `k` dała lepszy wynik?

To wszystko. Masz gotowy mini-projekt EDA + wizualizacja + model.

In [ ]:
# TODO końcowe: Uzupełnij listę wartości k i uruchom.
wyniki = []
for k in [3, 5, 7]:  # TODO: możesz zmienić listę
    m = KNeighborsClassifier(n_neighbors=k)
    m.fit(X_train, y_train)
    pred = m.predict(X_test)
    wyniki.append((k, accuracy_score(y_test, pred)))

pd.DataFrame(wyniki, columns=['k', 'accuracy']).sort_values('accuracy', ascending=False)